## Hypothesis testing from the beginning

### 1. Why do we need it?

Your 86,637 bookings are a **sample** of what the hotels' guests do. We want conclusions that hold in general, not only in this dataset. But samples are affected by luck. Here is the problem:

> City Hotel cancels 30.2% and Resort Hotel 23.7%. Is that a **real difference between the hotels**, or just **random variation** that would shrink if we looked at different bookings?

Hypothesis testing is the method that answers this fairly, with a number instead of a guess.

**A coin example.** You flip a coin and want to know if it is biased:

| Result | Chance of getting this many heads or more with a **fair** coin | Verdict |
|---|---|---|
| 6 heads out of 10 | 37.7% | Ordinary luck |
| 60 out of 100 | 2.8% | Suspicious |
| 600 out of 1000 | 0.000000014% (1.4e-10) | The coin is almost certainly biased |

The **same 60% heads** looks normal or impossible depending on how much data you have. A hypothesis test measures exactly that.

### 2. What is a hypothesis?

A hypothesis is a **testable claim about the population**, written as two competing statements:

| | Name | Meaning | Our hotel example |
|---|---|---|---|
| **H0** | **Null hypothesis** | Nothing is going on: no difference, no effect, no relationship | Both hotels have the same cancellation rate |
| **H1** | **Alternative hypothesis** | The claim we suspect is true | The rates are different |

**Why start by assuming H0?** It works like a court case: the defendant is **innocent until proven guilty**. We assume "no effect" and only reject it if the evidence is strong.

We don't prove H0, and we never say "we accept H0". We either:

- **Reject H0** â†’ enough evidence
- **Fail to reject H0** â†’ not enough evidence

### 3. The idea behind every test: the null distribution

1. **Imagine the world where H0 is true.** Even there, two samples won't match exactly. Luck makes them differ a little.
2. Take one number that summarises our data's gap, the **test statistic** (Z, t, F or chi-square).
3. In the H0 world, that statistic would take many values. Most are near the middle, and a few are extreme. That spread is the **null distribution**.
4. Ask: **where does our real number land on that curve?**
   - In the crowded middle â†’ ordinary luck, so we can't reject H0.
   - In the far tail â†’ luck is an unlikely explanation, so we reject H0.
5. The **p-value** is the probability of getting a result **at least this extreme if H0 were true**.

**p-value, in plain words:**

> "If the hotels really had identical cancellation rates, how often would a sample show a gap this big?"

Our answer was **2.77e-96**, so essentially never.

### 4. The rule for deciding: alpha

We set a cut-off before testing, the **significance level Î± = 0.05**:

- **p < 0.05** â†’ reject H0 â†’ statistically significant
- **p â‰¥ 0.05** â†’ fail to reject H0 â†’ not statistically significant

There are two ways to be wrong:

| | H0 is actually true | H0 is actually false |
|---|---|---|
| **We reject H0** | âŒ **Type I error** (false alarm). Chance = Î± = 5% | âœ… Correct |
| **We don't reject H0** | âœ… Correct | âŒ **Type II error** (missed a real effect) |

Choosing Î± = 0.05 means we accept a **5% risk of a false alarm**.

### 5. One-sided vs two-sided

- **Two-sided:** "Are they **different**?" H1 uses **â‰ **. The 5% rejection zone is split across both tails.  
  â†’ **H1: Hotels**

- **One-sided:** "Is A **higher** than B?" H1 uses **>** or **<**. The whole 5% goes into one tail.  
  â†’ **H3: Online TA**

Decide this **before** you look at the results, never after.

### 6. Statistically significant is not the same as important

With 86,637 bookings, even a tiny difference can get a very small p-value.

So for every test we also report the **effect size** â€” the actual size of the difference.

Examples:

- 6.5 percentage points
- 35 days
- Cohen's d

The simple idea is:

> **p-value â†’ Is the difference statistically real?**  
> **Effect size â†’ Is the difference large enough to matter?**

### 7. How to pick the test: your whiteboard tree

Ask three questions, in this order:

1. **What kind of data?**
   - Proportion (%)
   - Counts in categories
   - Mean
   - Variance

2. **How many groups?**
   - 1 group
   - 2 groups
   - More than 2 groups

3. **Any extra detail?**
   - Independent or paired groups
   - Number of factors
   - Other assumptions

### 8. Our 5 hypotheses, in plain words

| # | Business question | H0 (nothing going on) | H1 (what we suspect) | Test |
|---|---|---|---|---|
| **H1** | Do hotels differ in cancellations? | Same cancellation rate | Different rate | **2 prop test** |
| **H2** | Do cancelled bookings book earlier? | Same average lead time | Cancelled have longer average lead time | **2 sample t test** (after a **2 var test**) |
| **H3** | Is Online TA riskier? | Same cancellation rate as other channels | Online TA's rate is higher | **2 prop test** (one-sided) |
| **H4** | Does deposit type relate to cancellation? | Deposit type and cancellation are unrelated | They are related | **Degree of association (chi-square)** |
| **H5** | Does price differ by season? | All seasons have the same average ADR | At least one season differs | **One-way ANOVA** |

### Each test follows the same 5-step recipe

1. State **H0 and H1**.
2. Choose **Î± = 0.05** and the test.
3. Compute the **test statistic and p-value**.
4. Report the **effect size**.
5. Make the decision and explain the **business meaning**.

### 9. The notebook plan (`04_hypothesis_testing.ipynb`)

| Cell | Content |
|---|---|
| 1 | Setup |
| 2 | **Intuition demo:** See the null distribution built from your own data |
| 3 | H1 â€” 2 prop test |
| 4 | H2 â€” 2 var test, then 2 sample t test |
| 5 | H3 â€” 2 prop test, one-sided |
| 6 | H4 â€” Degree of association |
| 7 | H5 â€” One-way ANOVA |
| 8 | Summary table of all results |

If you already have old cells in this notebook, delete them and start fresh with this order.

You can reuse the **H1, H2 and H3 code** already provided because those tests are already part of your hypothesis-testing plan.

In [4]:
import pandas as pd
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest
from pathlib import Path

ROOT = Path.cwd().parent
DATA_PATH = ROOT / "data" / "processed" / "hotel_bookings_clean.csv"

df = pd.read_csv(DATA_PATH)

# our rule: if p-value is below alpha, we reject H0
alpha = 0.05

print(df.shape)
print("Significance level (alpha):", alpha)

(86637, 42)
Significance level (alpha): 0.05


In [13]:
# H0: both hotels have the SAME cancellation rate
# H1: the rates are DIFFERENT
# Test: 2 prop test (Discrete -> Proportion -> 2 samples)


# Step 1: cancellations and total bookings for each hotel
counts = df.groupby("hotel")["is_canceled"].agg(["sum", "count"])
rates = counts["sum"] / counts["count"] * 100
print(rates.round(1))

# Step 2: run the test (Z follows a normal curve if H0 is true)
z,p = proportions_ztest(counts["sum"].values , counts["count"].values)
print("Z =", round(z, 2), "| p-value:", "< 0.05 = " ,round(p,1) if p < 0.05 else round(p, 3))
# Step 3: decision (reject H0 if p < 0.05)
print("Reject H0" if p < alpha else "Cannot reject H0")


hotel
City Hotel      30.2
Resort Hotel    23.7
dtype: float64
Z = 20.83 | p-value: < 0.05 =  0.0
Reject H0


In [17]:
# H0: cancelled and kept bookings have the SAME average lead time
# H1: cancelled bookings have a LONGER average lead time
# Test: 2 sample t test

# Step 1: make two piles of numbers
cancelled = df[df["is_canceled"] == 1]["lead_time"]
kept = df[df["is_canceled"] == 0]["lead_time"]

# Step 2: look at the average of each pile
print("Average lead time of cancelled bookings:", round(cancelled.mean(), 1))
print("Average lead time of kept bookings:", round(kept.mean(), 1))

# Step 3: run the test
t, p = stats.ttest_ind(cancelled, kept, equal_var=False, alternative="greater")
print("t =", round(t, 1))
print("p-value:", "< 0.001" if p < 0.001 else round(p, 3))

# Step 4: decide
print("Reject H0" if p < alpha else "Cannot reject H0")


Average lead time of cancelled bookings: 105.8
Average lead time of kept bookings: 70.5
t = 52.2
p-value: < 0.001
Reject H0


Cancelled bookings were made about 35 days earlier than kept bookings (106 vs 71 days). A 2 sample t test gives t = 52.2 and p < 0.001, so we reject H0. Early bookings are riskier

The question: Do bookings made through online travel agents (Booking.com, Expedia and similar) get cancelled more often than bookings from every other channel?

In [21]:
# H0: Online TA and other channels have the SAME cancellation rate
# H1: Online TA has a HIGHER cancellation rate
# Test: 2 prop test (one-sided)

# Step 1: make two piles of 0/1 values (1 = cancelled, 0 = kept)
online = df[df["market_segment"] == "Online TA"]["is_canceled"]
other = df[df["market_segment"] != "Online TA"]["is_canceled"]

# Step 2: look at the counts and the rates
print("Online TA: cancelled", online.sum(), "out of", len(online))
print("Other channels: cancelled", other.sum(), "out of", len(other))
print("Online TA cancellation rate (%):", round(online.mean() * 100, 1))
print("Other channels cancellation rate (%):", round(other.mean() * 100, 1))

# Step 3: run the test
z, p = proportions_ztest([online.sum(), other.sum()], [len(online), len(other)], alternative="larger")
print("Z =", round(z, 1))
print("p-value:", "< 0.001" if p < 0.001 else round(p, 3))

# Step 4: decide
print("Reject H0" if p < alpha else "Cannot reject H0")

Online TA: cancelled 18230 out of 51285
Other channels: cancelled 5755 out of 35352
Online TA cancellation rate (%): 35.5
Other channels cancellation rate (%): 16.3
Z = 62.3
p-value: < 0.001
Reject H0


Online TA bookings are cancelled significantly more often than bookings from other channels (35.5% vs 16.3%). A one-sided 2 prop test gives Z = 62.3 and p < 0.001, so we reject H0. Online TA is the highest-risk channel and accounts for 59% of all bookings.

The question: Is a booking's deposit type (No Deposit, Non Refund, Refundable) connected to whether it gets cancelled?

In [22]:
# H0: deposit type and cancellation are NOT related
# H1: deposit type and cancellation ARE related
# Test: Degree of association (chi-square), 2 categorical variables

# Step 1: table of counts (rows = deposit type, columns = kept 0 / cancelled 1)
table = pd.crosstab(df["deposit_type"], df["is_canceled"])
print(table)

# Step 2: cancellation rate for each deposit type
print((df.groupby("deposit_type")["is_canceled"].mean() * 100).round(1))

# Step 3: run the test
chi2, p, dof, expected = stats.chi2_contingency(table)
print("Chi-square =", round(chi2, 1), "| degrees of freedom =", dof)
print("p-value:", "< 0.001" if p < 0.001 else round(p, 3))
print("Smallest expected count:", round(expected.min(), 1))

# Step 4: how strong is the relationship? (0 = none, 1 = perfect)
strength = (chi2 / len(df)) ** 0.5
print("Strength of relationship (0 = none, 1 = perfect):", round(strength, 2))

# Step 5: decide
print("Reject H0" if p < alpha else "Cannot reject H0")

is_canceled       0      1
deposit_type              
No Deposit    62516  22977
Non Refund       55    982
Refundable       81     26
deposit_type
No Deposit    26.9
Non Refund    94.7
Refundable    24.3
Name: is_canceled, dtype: float64
Chi-square = 2354.6 | degrees of freedom = 2
p-value: < 0.001
Smallest expected count: 29.6
Strength of relationship (0 = none, 1 = perfect): 0.16
Reject H0


The surprise: Non Refund bookings are cancelled 94.7% of the time. You'd expect the opposite, because guests who can't get their money back should show up. The test only tells us the two columns are related. It does not say a non-refundable deposit causes cancellations. The likely story is something about how those bookings were made, which we can't see in the data. Write this as a limitation in your report.

The strength is only 0.16 (weak-to-moderate). Deposit type is related to cancellation, but it doesn't decide it on its own. Also, only 1.2% of bookings are Non Refund, so this finding affects few bookings.

Deposit type is significantly related to cancellation (chi-square = 2354.6, df = 2, p < 0.001; strength = 0.16). We reject H0. Non Refund bookings are cancelled 94.7% of the time versus 26.9% for No Deposit, but this is an association, not proof that deposits cause cancellations

The question: Does the average nightly price (ADR) differ across the four seasons, or is it about the same all year?

In [23]:
# H0: the average ADR is the SAME in all four seasons
# H1: at least one season has a DIFFERENT average ADR
# Test: One-way ANOVA (Continuous mean -> more than 2 samples -> 1 factor)

# Step 1: the average price and number of bookings in each season
print(df.groupby("season")["adr"].agg(["mean", "count"]).round(1))

# Step 2: make four piles of prices, one per season
winter = df[df["season"] == "Winter"]["adr"]
spring = df[df["season"] == "Spring"]["adr"]
summer = df[df["season"] == "Summer"]["adr"]
autumn = df[df["season"] == "Autumn"]["adr"]

# Step 3: run the test
f, p = stats.f_oneway(winter, spring, summer, autumn)
print("F =", round(f, 1))
print("p-value:", "< 0.001" if p < 0.001 else round(p, 3))

# Step 4: how much of the price variation does season explain?
k = 4                                    # number of groups (seasons)
share = (f * (k - 1)) / (f * (k - 1) + (len(df) - k))
print("Share of price variation explained by season (%):", round(share * 100, 1))

# Step 5: decide
print("Reject H0" if p < alpha else "Cannot reject H0")

         mean  count
season              
Autumn   94.4  18417
Spring   99.8  23604
Summer  138.1  28903
Winter   76.5  15713
F = 7436.2
p-value: < 0.001
Share of price variation explained by season (%): 20.5
Reject H0


Average ADR differs significantly across seasons (one-way ANOVA, F = 7436.2, p < 0.001). Summer is the most expensive (€138.1) and winter the cheapest (€76.5). Season explains about 20.5% of price variation. We reject H0.

In [24]:
# Two small helpers so we don't repeat ourselves
def show_p(p):
    return "< 0.001" if p < 0.001 else round(p, 3)

def decide(p):
    return "Reject H0" if p < alpha else "Cannot reject H0"

results = []          # an empty list; each test adds one row to it

# ---- H1: hotels (2 prop test) ----
counts = df.groupby("hotel")["is_canceled"].agg(["sum", "count"])
rates = counts["sum"] / counts["count"] * 100
z, p = proportions_ztest(counts["sum"].values, counts["count"].values)
results.append({"Hypothesis": "H1: Hotels differ in cancellation rate",
                "Test": "2 prop test", "Statistic": round(z, 1), "p-value": show_p(p),
                "Effect": f"City {rates['City Hotel']:.1f}% vs Resort {rates['Resort Hotel']:.1f}%",
                "Decision": decide(p)})

# ---- H2: lead time (2 sample t test) ----
cancelled = df[df["is_canceled"] == 1]["lead_time"]
kept = df[df["is_canceled"] == 0]["lead_time"]
t, p = stats.ttest_ind(cancelled, kept, equal_var=False, alternative="greater")
results.append({"Hypothesis": "H2: Cancelled bookings have longer lead time",
                "Test": "2 sample t test", "Statistic": round(t, 1), "p-value": show_p(p),
                "Effect": f"{cancelled.mean():.1f} vs {kept.mean():.1f} days",
                "Decision": decide(p)})

# ---- H3: Online TA (2 prop test, one-sided) ----
online = df[df["market_segment"] == "Online TA"]["is_canceled"]
other = df[df["market_segment"] != "Online TA"]["is_canceled"]
z, p = proportions_ztest([online.sum(), other.sum()], [len(online), len(other)], alternative="larger")
results.append({"Hypothesis": "H3: Online TA cancels more than other channels",
                "Test": "2 prop test (one-sided)", "Statistic": round(z, 1), "p-value": show_p(p),
                "Effect": f"Online TA {online.mean()*100:.1f}% vs Other {other.mean()*100:.1f}%",
                "Decision": decide(p)})

# ---- H4: deposit type (degree of association) ----
table = pd.crosstab(df["deposit_type"], df["is_canceled"])
chi2, p, dof, expected = stats.chi2_contingency(table)
dep = df.groupby("deposit_type")["is_canceled"].mean() * 100
results.append({"Hypothesis": "H4: Deposit type related to cancellation",
                "Test": "Degree of association", "Statistic": round(chi2, 1), "p-value": show_p(p),
                "Effect": f"Non Refund {dep['Non Refund']:.1f}% vs No Deposit {dep['No Deposit']:.1f}%",
                "Decision": decide(p)})

# ---- H5: seasons (one-way ANOVA) ----
piles = [df[df["season"] == s]["adr"] for s in ["Winter", "Spring", "Summer", "Autumn"]]
f, p = stats.f_oneway(*piles)
adr = df.groupby("season")["adr"].mean()
results.append({"Hypothesis": "H5: Average ADR differs across seasons",
                "Test": "One-way ANOVA", "Statistic": round(f, 1), "p-value": show_p(p),
                "Effect": f"Summer {adr['Summer']:.1f} vs Winter {adr['Winter']:.1f}",
                "Decision": decide(p)})

# ---- turn the list into a table, show it, and save it ----
summary = pd.DataFrame(results)
print(summary.to_string(index=False))

REPORTS_DIR = ROOT / "reports"
REPORTS_DIR.mkdir(exist_ok=True)
summary.to_csv(REPORTS_DIR / "hypothesis_test_summary.csv", index=False)
print("Saved to reports/hypothesis_test_summary.csv")

                                    Hypothesis                    Test  Statistic p-value                               Effect  Decision
        H1: Hotels differ in cancellation rate             2 prop test       20.8 < 0.001           City 30.2% vs Resort 23.7% Reject H0
  H2: Cancelled bookings have longer lead time         2 sample t test       52.2 < 0.001                   105.8 vs 70.5 days Reject H0
H3: Online TA cancels more than other channels 2 prop test (one-sided)       62.3 < 0.001       Online TA 35.5% vs Other 16.3% Reject H0
      H4: Deposit type related to cancellation   Degree of association     2354.6 < 0.001 Non Refund 94.7% vs No Deposit 26.9% Reject H0
        H5: Average ADR differs across seasons           One-way ANOVA     7436.2 < 0.001          Summer 138.1 vs Winter 76.5 Reject H0
Saved to reports/hypothesis_test_summary.csv
